# GVH Diagonal Cubic 0.3.2.7.3.7.3.1
## Lapse-Gradient Canonicalization and True Secondary Normal Constraint Extraction

**Auteur :** Charlemagne O Laurince

### Cible unique

Partir du verrou découvert dans `7.7.3` :
\[
a_i^{(n)}=D_i\ln N=\frac{D_iN}{N},
\]
et extraire correctement
\[
\boxed{
\mathscr C_N=-\frac{\delta H_C}{\delta N}
=-\frac{\partial H_C}{\partial N}
+D_i\!\left(\frac{\partial H_C}{\partial(D_iN)}\right).
}
\]

Le notebook distingue strictement :
- la transformée de Legendre déjà obtenue ;
- la vraie contrainte secondaire issue de la variation fonctionnelle du lapse ;
- la question ultérieure de fermeture de l'algèbre.

Aucun `RDD3=True` ne sera déclaré ici.


In [1]:
import sympy as sp, json, sys
from pathlib import Path
print("GVH 0.3.2.7.3.7.3.1")
print("Python:",sys.version.split()[0])
print("SymPy:",sp.__version__)


GVH 0.3.2.7.3.7.3.1
Python: 3.12.13
SymPy: 1.14.0


## 1. Identité variationnelle locale

Pour une densité hamiltonienne locale
\[
\mathcal H_C=N\,F(a_i),\qquad a_i=\frac{N_i}{N},\quad N_i\equiv D_iN,
\]
on traite \(N\) et \(N_i\) comme variables de jet indépendantes lors de l'opération d'Euler–Lagrange.

On obtient :
\[
\frac{\partial \mathcal H_C}{\partial N}
=F-a_iF_{,a_i},
\qquad
\frac{\partial \mathcal H_C}{\partial N_i}=F_{,a_i}.
\]

Donc
\[
\boxed{
\mathscr C_N
=-F+a_iF_{,a_i}+D_iF_{,a_i}.
}
\]

C'est la forme canonique correcte avant toute simplification dynamique.


In [2]:
N=sp.symbols("N", positive=True)
N1,N2,N3=sp.symbols("N1 N2 N3", real=True)
a1,a2,a3=sp.symbols("a1 a2 a3", real=True)
a=sp.Matrix([a1,a2,a3])
Ni=sp.Matrix([N1,N2,N3])

# Generic polynomial witness for F(a), with coefficients independent of N.
f0=sp.symbols("f0", real=True)
b1,b2,b3=sp.symbols("b1 b2 b3", real=True)
g11,g22,g33,g12,g13,g23=sp.symbols("g11 g22 g33 g12 g13 g23", real=True)
G=sp.Matrix([[g11,g12,g13],[g12,g22,g23],[g13,g23,g33]])
b=sp.Matrix([b1,b2,b3])
F=sp.expand(f0+(b.T*a)[0]+sp.Rational(1,2)*(a.T*G*a)[0])

H=sp.expand(N*F.subs({a1:N1/N,a2:N2/N,a3:N3/N}))
dH_dN=sp.simplify(sp.diff(H,N))
dH_dNi=sp.Matrix([sp.simplify(sp.diff(H,x)) for x in Ni])

expected_dN=sp.simplify(F-(a.T*sp.Matrix([sp.diff(F,x) for x in a]))[0])
expected_dNi=sp.Matrix([sp.diff(F,x) for x in a])

subs_back={N1:N*a1,N2:N*a2,N3:N*a3}
assert sp.simplify(dH_dN.subs(subs_back)-expected_dN)==0
assert all(sp.simplify(dH_dNi[i].subs(subs_back)-expected_dNi[i])==0 for i in range(3))

print("Jet-space lapse variation identity: PASS")


Jet-space lapse variation identity: PASS


## 2. Application à la structure héritée de 7.7.2

Écrivons la transformée de Legendre locale sous la forme
\[
F(a)=
\frac12(P-J(a))^TQ^{-1}(P-J(a))-U(a).
\]

Dans le secteur étudié, \(Q\) est indépendant de \(a_i^{(n)}\), tandis que \(J\) et \(U\) en dépendent.

La dérivée exacte est donc
\[
F_{,a_i}
=
-\left(\frac{\partial J}{\partial a_i}\right)^T
Q^{-1}(P-J)
-\frac{\partial U}{\partial a_i}.
\]

Ainsi la vraie contrainte secondaire est
\[
\boxed{
\mathscr C_N
=
-F+a_iF_{,a_i}+D_iF_{,a_i}.
}
\]


In [3]:
# Symbolic finite-dimensional representation of the exact formula.
m=4
P=sp.Matrix(sp.symbols("P0:4"))
avec=sp.Matrix(sp.symbols("aa0:3"))
Qinv=sp.Matrix(4,4,sp.symbols("q0:16"))
Qinv=(Qinv+Qinv.T)/2

# affine J(a), quadratic U(a): enough to verify the chain-rule identity exactly.
J0=sp.Matrix(sp.symbols("j0:4"))
L=sp.Matrix(4,3,sp.symbols("l0:12"))
u0=sp.symbols("u0")
ub=sp.Matrix(sp.symbols("ub0:3"))
UG=sp.Matrix(3,3,sp.symbols("ug0:9")); UG=(UG+UG.T)/2

J=J0+L*avec
U=u0+(ub.T*avec)[0]+sp.Rational(1,2)*(avec.T*UG*avec)[0]
Pi=P-J
Fcanon=sp.expand(sp.Rational(1,2)*(Pi.T*Qinv*Pi)[0]-U)

Fa_direct=sp.Matrix([sp.diff(Fcanon,x) for x in avec])
Fa_chain=sp.Matrix([
    -(L[:,i].T*Qinv*Pi)[0]-sp.diff(U,avec[i])
    for i in range(3)
])
assert all(sp.expand(Fa_direct[i]-Fa_chain[i])==0 for i in range(3))
print("Canonical F_,a chain rule: PASS")


Canonical F_,a chain rule: PASS


## 3. Le terme divergence est essentiel

Le terme
\[
D_iF_{,a_i}
\]
n'est pas optionnel : il contient les dérivées spatiales des variables canoniques et des champs.

On introduit donc la notation exacte
\[
\mathcal B^i\equiv F_{,a_i},
\]
puis
\[
\boxed{
\mathscr C_N=-F+a_i\mathcal B^i+D_i\mathcal B^i.
}
\]

Cette écriture isole complètement la dépendance fonctionnelle du lapse : \(N\) et \(D_iN\) n'apparaissent plus explicitement en dehors de \(a_i\), et la prochaine question est de savoir si les termes en \(a_i\) s'annulent ou se réorganisent après développement complet de \(\mathcal B^i\).


In [4]:
Bvec=Fa_chain
CN_nodiv=sp.expand(-Fcanon+(avec.T*Bvec)[0])

print("B^i = dF/da_i constructed: PASS")
print("C_N local non-divergence part constructed: PASS")
print("D_i B^i retained as functional divergence operator: PASS")


B^i = dF/da_i constructed: PASS
C_N local non-divergence part constructed: PASS
D_i B^i retained as functional divergence operator: PASS


## 4. Critère d'indépendance du lapse

Pour pouvoir écrire le Hamiltonien sous la forme standard
\[
H_C=\int d^3x\left(N\mathscr C_N+N^i\mathscr C_i+\cdots\right),
\]
la densité secondaire finale doit être indépendante du multiplicateur \(N\) et de ses gradients, éventuellement **modulo contraintes auxiliaires**.

Dans la représentation générique \(F(a)\), la présence de
\[
a_i\mathcal B^i
\]
montre qu'une annulation n'est pas automatique.

Il faut donc effectuer cette vérification avec les \(J(a)\) et \(U(a)\) complets du modèle avant de promouvoir \(\mathscr C_N\) en contrainte hypersurface standard.


In [5]:
# Demonstrate non-automatic cancellation on a simple exact witness.
aa0,aa1,aa2=avec
witness={
    **{P[i]:sp.Rational(i+2,3) for i in range(4)},
    **{J0[i]:sp.Rational(i+1,7) for i in range(4)},
    **{L[i,j]:sp.Rational((i+1)*(j+1),19) for i in range(4) for j in range(3)},
    u0:sp.Rational(2,11),
    **{ub[i]:sp.Rational(i+1,13) for i in range(3)},
}
# Set Qinv=identity and UG=0.
for i in range(4):
    for j in range(4):
        # original q symbols occur symmetrized; substitute all base q's.
        witness[sp.symbols("q0:16")[4*i+j]]=1 if i==j else 0
for x in sp.symbols("ug0:9"):
    witness[x]=0

expr=sp.simplify(CN_nodiv.subs(witness))
depends=any(expr.has(x) for x in avec)
assert depends
print("Generic cancellation of a_i dependence is NOT automatic: PASS")


Generic cancellation of a_i dependence is NOT automatic: PASS


## 5. Verdict causal

Cette étape réussit l'opération mathématique centrale :
\[
\boxed{
-\frac{\delta H_C}{\delta N}
=
-F+a_iF_{,a_i}+D_iF_{,a_i}
}
\]
pour la structure locale \(H_C=N F(DN/N)\).

Elle donne aussi explicitement
\[
F_{,a_i}
=
-\left(\frac{\partial J}{\partial a_i}\right)^TQ^{-1}(P-J)
-\frac{\partial U}{\partial a_i}.
\]

Mais elle montre qu'une contrainte normale indépendante du lapse **ne découle pas automatiquement** de la seule transformée de Legendre.

Il reste à injecter les expressions complètes \(J_u(a)\), \(U_u(a)\), puis à calculer le terme spatial
\[
D_iF_{,a_i}
\]
avec les dérivées covariantes complètes afin de tester l'annulation effective des \(a_i\).


In [6]:
GATES={
 "lapse_jet_variation_identity_exact":True,
 "canonical_F_ai_chain_rule_exact":True,
 "true_secondary_normal_constraint_operator_derived":True,
 "divergence_term_identified":True,
 "generic_ai_cancellation_not_automatic":True,
 "full_model_covariant_divergence_expanded":False,
 "final_CN_independent_of_lapse_gradient":False,
 "true_secondary_normal_constraint_fully_reduced":False,
 "RDD3_computed":False,
 "hypersurface_algebra_closed":False,
}
for k,v in GATES.items():
    print(k,":",v)

FINAL_STATUS=(
 "PARTIAL-PASS-EXACT-LAPSE-FUNCTIONAL-VARIATION-DERIVED_"
 "TRUE-NORMAL-CONSTRAINT-OPERATOR-OBTAINED_"
 "BLOCKED-FULL-COVARIANT-DIVERGENCE-AND-AI-CANCELLATION-TEST"
)
DISPERSION_READY=False
print("\nFINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


lapse_jet_variation_identity_exact : True
canonical_F_ai_chain_rule_exact : True
true_secondary_normal_constraint_operator_derived : True
divergence_term_identified : True
generic_ai_cancellation_not_automatic : True
full_model_covariant_divergence_expanded : False
final_CN_independent_of_lapse_gradient : False
true_secondary_normal_constraint_fully_reduced : False
RDD3_computed : False
hypersurface_algebra_closed : False

FINAL STATUS: PARTIAL-PASS-EXACT-LAPSE-FUNCTIONAL-VARIATION-DERIVED_TRUE-NORMAL-CONSTRAINT-OPERATOR-OBTAINED_BLOCKED-FULL-COVARIANT-DIVERGENCE-AND-AI-CANCELLATION-TEST
DISPERSION_READY = False


## 6. Prochaine sous-étape

La prochaine étape logique est :

\[
\boxed{\mathbf{0.3.2.7.3.7.3.2}}
\]

### Full Covariant Divergence Expansion and Lapse-Gradient Cancellation Test

Elle devra :
1. réinjecter les \(J_u(a)\) et \(U_u(a)\) complets ;
2. construire \(\mathcal B^i=F_{,a_i}\) sans modèle réduit ;
3. développer \(D_i\mathcal B^i\) avec la géométrie spatiale complète ;
4. former
   \[
   \mathscr C_N=-F+a_i\mathcal B^i+D_i\mathcal B^i;
   \]
5. tester si toute dépendance en \(a_i=D_i\ln N\) disparaît, directement ou modulo contraintes auxiliaires.

Seulement après ce test, on pourra revenir aux crochets de 7.7.3.


In [7]:
artifact={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.1",
 "final_status":FINAL_STATUS,
 "CN_operator":"-F + a_i*dF/da_i + D_i(dF/da_i)",
 "F_ai":"-(dJ/da_i)^T Q^{-1}(P-J) - dU/da_i",
 "gates":GATES,
 "dispersion_ready":False,
 "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.2_Full_Covariant_Divergence_Expansion_and_Lapse_Gradient_Cancellation_Test.ipynb"
}
export_dir=Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/"gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
p=export_dir/"gvh_0.3.2.7.3.7.3.1_lapse_gradient_canonicalization.json"
p.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",p)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.3.1_lapse_gradient_canonicalization.json


# Conclusion

L'étape `7.7.3.1` ne ferme pas artificiellement \(R_{DD3}\).

Elle établit exactement la bonne opération de variation du lapse et transforme le verrou découvert par `7.7.3` en une cible calculable :

\[
\boxed{
\mathscr C_N
=
-F+a_i\mathcal B^i+D_i\mathcal B^i,
\qquad
\mathcal B^i=F_{,a_i}.
}
\]

Le verrou restant est désormais :
\[
\boxed{
D_i\mathcal B^i
\ \text{complet}
\quad+\quad
\text{test d'annulation de }a_i.
}
\]

Donc :
\[
\boxed{R_{DD3}\text{ reste OPEN}}
\]
et
\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
